# Movimiento de Inventario - DanuStore · Tablero en Streamlit
**Nexxus Consulting · Equipo 6 · TI3002C**

Corre las celdas **en orden**. El tablero tiene dos pestañas: **Plan de acción** (predictivo, red neuronal de la Etapa 5.1) y **Diagnóstico** (descriptivo).

| Paso | Qué hace |
|---|---|
| 1 | Instala librerías |
| 2 | Trae la data (y el logo de Danu) desde Google Drive |
| 3 | Escribe el código de la app en `app.py` |
| 4 | Revisa que todo esté listo |
| 5 | Lanza la app y abre el túnel |

## 1. Instalar librerías

In [ ]:
!pip install -q -U streamlit plotly

## 2. Traer la data desde Google Drive
Ajusta las rutas si moviste los archivos. Si no se encuentra el CSV, la celda te pedirá subirlo manualmente. El logo es opcional.

In [ ]:
from google.colab import drive
import os, shutil

drive.mount("/content/drive")

RUTA_DATA = "/content/drive/MyDrive/DanuStore Equipo 6/data_DanuStore_Equipo6_Mensual.csv"
RUTA_LOGO = "/content/drive/MyDrive/7mo/logo_danu.png"      # opcional

if os.path.exists(RUTA_DATA):
    shutil.copy(RUTA_DATA, "/content/data_DanuStore_Equipo6_Mensual.csv")
    print("✔ Data copiada desde Drive")
else:
    from google.colab import files
    print("No se encontró la data en Drive. Sube data_DanuStore_Equipo6_Mensual.csv:")
    files.upload()

if os.path.exists(RUTA_LOGO):
    shutil.copy(RUTA_LOGO, "/content/logo_danu.png")
    print("✔ Logo de Danu copiado")
else:
    print("ℹ Logo de Danu no encontrado: la app funciona sin él")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
No se encontró la data en Drive. Sube data_DanuStore_Equipo6_Mensual.csv:


## 3. Código de la app (`app.py`)
Esta celda **escribe** el archivo; no muestra nada más que *Writing app.py*. Si cambias el código, vuelve a correr esta celda y la del paso 5.

In [ ]:
%%writefile app.py
"""
Movimiento de Inventario - DanuStore  ·  v2 (Plan de acción + Diagnóstico)
Elaborado por Nexxus Consulting · Equipo 6 · TI3002C

Datos : data_DanuStore_Equipo6_Mensual.csv (en la misma carpeta que app.py)
Logo  : logo_danu.png (opcional, en la misma carpeta)
Modelo: Red neuronal MLP de la Etapa 5.1
        StandardScaler + MLP(64), alpha=0.001, learning_rate_init=0.001,
        partición 70/30 estratificada, random_state=42.
        Objetivo: nivel de sobrestock del trimestre siguiente (t+1…t+3) en 4 niveles por cuartiles.

Ejecutar:  streamlit run app.py
"""
import base64
import html
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import streamlit as st
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

st.set_page_config(page_title="Movimiento de Inventario - DanuStore", layout="wide")

CARPETA = Path(__file__).parent
ARCHIVO = CARPETA / "data_DanuStore_Equipo6_Mensual.csv"
LOGO_DANU = CARPETA / "logo_danu.png"
RANDOM_STATE = 42

# ──────────────────────────────────────────────────────────────────────────────
# Paleta y catálogos
# ──────────────────────────────────────────────────────────────────────────────
NAVY, GRIS, TINTA = "#1F3A5F", "#5B6673", "#1D2733"
NIVELES = ["Bajo", "Medio", "Alto", "Crítico"]
COL_NIVEL = ["#2E9E5B", "#E3B23C", "#E07A2E", "#C0392B"]
PILL = [("#DFF2E7", "#1E6B41"), ("#FDF1D6", "#7A5A00"), ("#FCE6D6", "#9A4A12"), ("#FBE3E0", "#A32D21")]
ACCIONES = ["Reducir compras", "Transferir", "Reabastecer", "Mantener"]
COL_ACCION = ["#B03A2E", "#D9822B", "#2E9E5B", "#8A96A3"]
NIVELES_ACCION = [[2, 3], [3], [0], [0, 1]]
QUE_HACER = ["Pausar o reducir el próximo pedido y activar promociones.",
             "Mover inventario a regiones de mayor rotación antes de comprar.",
             "Asegurar el próximo pedido para evitar faltantes.",
             "Conservar el nivel de compra actual y monitorear."]
DEMANDA = ["Baja", "Media", "Alta"]

MESES_EN = ["January", "February", "March", "April", "May", "June", "July",
            "August", "September", "October", "November", "December"]
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
REGION_ES = {"Zona Metropolitana de Guadalajara": "ZM Guadalajara", "Zona Metropolitana de Monterrey": "ZM Monterrey"}
CATEGORIA_ES = {"Accessories": "Accesorios", "Casual Wear": "Ropa casual", "Fitness Wear": "Ropa fitness", "Footwear": "Calzado"}
ORDEN_CATEGORIA = ["Calzado", "Accesorios", "Ropa casual", "Ropa fitness"]
SUBCAT_ES = {"Bags": "Bolsas", "Belts": "Cinturones", "Boots": "Botas", "Bottoms": "Bottoms", "Dresses": "Vestidos",
             "Hats": "Sombreros", "Jewelry": "Joyería", "Outerwear": "Chamarras", "Pants": "Pantalones",
             "Sandals": "Sandalias", "Shirts": "Camisas", "Sneakers": "Tenis", "Tops": "Tops"}

ICONOS = {
    "check": '<path d="M9 11l3 3 8-8"/><path d="M20 12v7a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2V5a2 2 0 0 1 2-2h11"/>',
    "pin": '<path d="M12 21s-7-6.2-7-11a7 7 0 0 1 14 0c0 4.8-7 11-7 11z"/><circle cx="12" cy="10" r="2.5"/>',
    "search": '<circle cx="11" cy="11" r="7"/><path d="M20 20l-3.5-3.5"/>',
    "alert": '<path d="M12 3.5l9 16H3z"/><path d="M12 10v4"/><path d="M12 17.2v.3"/>',
    "layers": '<path d="M12 3l9 5-9 5-9-5z"/><path d="M3 13l9 5 9-5"/>',
    "trend": '<path d="M3 17l6-6 4 4 8-8"/><path d="M15 7h6v6"/>',
    "compass": '<circle cx="12" cy="12" r="9"/><path d="M15.5 8.5l-2 5-5 2 2-5z"/>',
    "box": '<path d="M21 8l-9-5-9 5v8l9 5 9-5z"/><path d="M3 8l9 5 9-5"/><path d="M12 13v8"/>',
}

# ──────────────────────────────────────────────────────────────────────────────
# Estilo global (CSS)
# ──────────────────────────────────────────────────────────────────────────────
st.markdown(f"""
<style>
  [data-testid="stAppViewContainer"], [data-testid="stHeader"] {{ background: #F4F6F9; }}
  .block-container {{ padding-top: 3.4rem; max-width: 1320px; }}
  html, body, [class*="css"] {{ font-family: 'Segoe UI', Inter, system-ui, sans-serif; }}
  div[data-testid="stTabs"] button p {{ font-size: 15px; font-weight: 600; }}
  label p {{ font-size: 12px !important; font-weight: 600 !important; color: {GRIS} !important; }}
  div[data-testid="stVerticalBlockBorderWrapper"] {{ background: #FFFFFF; }}
  .stPlotlyChart {{ background: #FFFFFF; border: 1px solid #E3E7ED; border-radius: 12px; padding: 8px 6px 0; }}

</style>
""", unsafe_allow_html=True)


# ──────────────────────────────────────────────────────────────────────────────
# Datos y modelo
# ──────────────────────────────────────────────────────────────────────────────
@st.cache_data
def cargar_datos(ruta: Path) -> pd.DataFrame:
    df = pd.read_csv(ruta)
    df["mes"] = df["month"].map({m: i for i, m in enumerate(MESES_EN, 1)})
    df["t"] = (df["year"] - 2024) * 12 + df["mes"] - 1                      # 0..23
    df["Región"] = df["region"].replace(REGION_ES)
    df["Categoría"] = df["category"].map(CATEGORIA_ES)
    df["Subcategoría"] = df["subcategory"].map(SUBCAT_ES)
    df["Producto"] = df["Subcategoría"] + " · P" + df["product_id"].astype(str)
    df["meses_inv"] = df["Dias_Cobertura_Inventario"] / 30.4
    df["valor_inv"] = df["Stock_Restante_Mensual"] * df["static_price"]   # inventario × precio de lista (un solo mes)
    return df.sort_values(["region", "product_id", "t"]).reset_index(drop=True)


@st.cache_resource
def entrenar_y_pronosticar(df: pd.DataFrame):
    d = df.copy()
    cortes = d["meses_inv"].quantile([.25, .5, .75]).values
    g = d.groupby(["region", "product_id"])["meses_inv"]
    d["meses_trim"] = pd.concat([g.shift(-h) for h in (1, 2, 3)], axis=1).mean(axis=1, skipna=False)
    d["nivel_real"] = pd.cut(d["meses_trim"], [-np.inf, *cortes, np.inf], labels=False)
    d["nivel_obs"] = pd.cut(d["meses_inv"], [-np.inf, *cortes, np.inf], labels=False).astype(int)

    num, cat = ["units_expected", "percentage", "static_price", "mes"], ["region", "subcategory"]
    X_all = pd.get_dummies(d[num + cat], columns=cat, drop_first=True).astype(float)
    con_obj = d["nivel_real"].notna()
    X_tr, _, y_tr, _ = train_test_split(X_all[con_obj], d.loc[con_obj, "nivel_real"].astype(int),
                                        test_size=0.30, random_state=RANDOM_STATE,
                                        stratify=d.loc[con_obj, "nivel_real"].astype(int))
    modelo = Pipeline([("scaler", StandardScaler()),
                       ("mlp", MLPClassifier(hidden_layer_sizes=(64,), alpha=1e-3, learning_rate_init=1e-3,
                                             activation="relu", solver="adam", max_iter=500, early_stopping=True,
                                             validation_fraction=0.1, n_iter_no_change=10,
                                             random_state=RANDOM_STATE))]).fit(X_tr, y_tr)
    prob = modelo.predict_proba(X_all)
    d["nivel_pred"] = prob.argmax(1)
    for k, n in enumerate(NIVELES):
        d[f"p_{n}"] = prob[:, k]
    cortes_dem = d.loc[con_obj, "units_expected"].quantile([1 / 3, 2 / 3]).values
    d["demanda"] = pd.cut(d["units_expected"], [-np.inf, *cortes_dem, np.inf], labels=False).astype(int)
    lv, dm = d["nivel_pred"], d["demanda"]
    d["accion"] = np.select([((lv >= 2) & (dm == 0)) | ((lv == 2) & (dm >= 1)), (lv == 3) & (dm >= 1), (lv == 0) & (dm == 2)],
                            [0, 1, 2], default=3)
    # Cobertura esperada del trimestre siguiente: Σ P(nivel k) × cobertura mediana observada en el nivel k
    tipica = d[con_obj].groupby("nivel_real")["meses_trim"].median().reindex(range(4)).values
    d["meses_fut"] = prob @ tipica
    return d, cortes, cortes_dem


# ──────────────────────────────────────────────────────────────────────────────
# Componentes HTML
# ──────────────────────────────────────────────────────────────────────────────
def esc(x) -> str:
    return html.escape(str(x))


def dinero(v: float) -> str:
    return f"${v / 1e6:,.1f} M" if v >= 1e6 else f"${v / 1e3:,.0f} K" if v >= 1e3 else f"${v:,.0f}"


def meses(dias: float) -> str:
    return f"{dias / 30.4:.1f} meses"


def pct(a, b) -> str:
    return f"{a / b * 100:.1f}%" if b else "—"


def etiqueta_mes(t: int) -> str:
    return f"{MESES[t % 12]} {2024 + t // 12}"


def pill(k: int, chico=False) -> str:
    bg, fg = PILL[k]
    pad, fs = ("1px 7px", 10) if chico else ("2px 10px", 12)
    return f'<span style="display:inline-block;padding:{pad};border-radius:10px;font-size:{fs}px;font-weight:700;background:{bg};color:{fg};">{NIVELES[k]}</span>'


def svg_img(svg: str, w: int, h: int) -> str:
    """st.html no muestra <svg> en línea: se incrusta como imagen."""
    b64 = base64.b64encode(svg.encode()).decode()
    return f'<img src="data:image/svg+xml;base64,{b64}" width="{w}" height="{h}" alt="" style="display:block;">'


def icono(nombre: str) -> str:
    return svg_img(f'<svg xmlns="http://www.w3.org/2000/svg" width="22" height="22" viewBox="0 0 24 24" fill="none" stroke="{NAVY}" '
                   f'stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round">{ICONOS[nombre]}</svg>', 22, 22)


LOGO_NEXXUS = ('<svg xmlns="http://www.w3.org/2000/svg" width="30" height="30" viewBox="0 0 30 30"><rect x="1" y="1" width="6" height="28" rx="1" fill="#FFFFFF"/>'
               '<rect x="23" y="1" width="6" height="28" rx="1" fill="#FFFFFF"/><polygon points="1,1 7,1 29,29 23,29" fill="#7FA8D6"/></svg>')


def punto(color: str, s: int = 9) -> str:
    return f'<span style="width:{s}px;height:{s}px;border-radius:50%;display:inline-block;flex:none;background:{color};"></span>'


def encabezado_seccion(num: str, icono_: str, titulo: str, sub: str):
    st.html(f"""
    <div style="display:flex;align-items:center;gap:20px;padding:16px 22px;margin:18px 0 10px;background:#FFFFFF;border:1px solid #E3E7ED;border-left:5px solid {NAVY};border-radius:12px;font-family:'Segoe UI',Inter,sans-serif;">
      <span style="font-size:36px;font-weight:300;line-height:1;color:#A9C4E4;">{num}</span>
      <span style="width:44px;height:44px;border-radius:10px;background:#EEF3F9;display:flex;align-items:center;justify-content:center;flex:none;">
        {icono(icono_)}</span>
      <div style="display:flex;flex-direction:column;gap:4px;">
        <span style="margin:0;font-size:20px;font-weight:700;color:{NAVY};">{esc(titulo)}</span>
        <span style="margin:0;font-size:13px;color:{GRIS};">{esc(sub)}</span>
      </div>
    </div>""")


def tarjeta(titulo: str, sub: str, cuerpo: str, alto: int | None = None) -> str:
    h = f"min-height:{alto}px;" if alto else ""
    return f"""
    <div style="{h}padding:18px 20px;background:#FFFFFF;border:1px solid #E3E7ED;border-radius:12px;display:flex;flex-direction:column;gap:14px;font-family:'Segoe UI',Inter,sans-serif;color:{TINTA};box-sizing:border-box;">
      <div style="display:flex;flex-direction:column;gap:3px;">
        <span style="font-size:15px;font-weight:700;">{esc(titulo)}</span>
        <span style="font-size:12px;color:{GRIS};">{esc(sub)}</span>
      </div>{cuerpo}
    </div>"""


def tabla(columnas, filas, alto_max: int | None = None, fuente: int = 12) -> str:
    """columnas: [(titulo, 'l'|'r')]; filas: listas de celdas HTML ya formateadas."""
    th = "padding:8px 10px;font-size:11px;font-weight:600;letter-spacing:.06em;text-transform:uppercase;color:#8A96A3;border-bottom:1px solid #E3E7ED;position:sticky;top:0;background:#FFFFFF;"
    td = "padding:8px 10px;border-bottom:1px solid #F0F2F5;font-variant-numeric:tabular-nums;white-space:nowrap;"
    cab = "".join(f'<th style="{th}text-align:{"right" if a == "r" else "left"};">{esc(c)}</th>' for c, a in columnas)
    cuerpo = "".join("<tr>" + "".join(
        f'<td style="{td}text-align:{"right" if a == "r" else "left"};">{celda}</td>' for celda, (_, a) in zip(fila, columnas)) + "</tr>"
                     for fila in filas)
    if not filas:
        cuerpo = f'<tr><td colspan="{len(columnas)}" style="{td}color:{GRIS};">Sin registros para los filtros seleccionados.</td></tr>'
    caja = f"max-height:{alto_max}px;overflow:auto;" if alto_max else "overflow-x:auto;"
    return f'<div style="{caja}"><table style="width:100%;border-collapse:collapse;font-size:{fuente}px;color:{TINTA};">' \
           f'<thead><tr>{cab}</tr></thead><tbody>{cuerpo}</tbody></table></div>'


def barras_h(items, color: str = NAVY) -> str:
    """items: [(etiqueta, valor, texto)] → barras horizontales de una sola medida."""
    mx = max([v for _, v, _ in items] + [1e-9])
    return '<div style="display:flex;flex-direction:column;gap:13px;">' + "".join(
        f'<div style="display:grid;grid-template-columns:124px 1fr 76px;gap:12px;align-items:center;font-size:13px;">'
        f'<span style="color:#3D4752;">{esc(n)}</span><div style="height:12px;background:#F0F2F5;border-radius:6px;">'
        f'<div style="height:12px;border-radius:6px;background:{color};width:{v / mx * 100:.1f}%;"></div></div>'
        f'<span style="text-align:right;font-weight:700;">{esc(txt)}</span></div>' for n, v, txt in items) + "</div>"


def barras_nivel(valores, textos, alto: int = 200, resaltar: int | None = None) -> str:
    mx = max(list(valores) + [1e-9])
    cols = "".join(
        f'<div style="flex:1;height:100%;display:flex;flex-direction:column;justify-content:flex-end;align-items:center;gap:6px;">'
        f'<span style="font-size:13px;font-weight:700;">{esc(t)}</span>'
        f'<div style="width:100%;max-width:88px;border-radius:6px 6px 0 0;background:{COL_NIVEL[k]};height:{max(1, v / mx * (alto - 30)):.0f}px;'
        f'opacity:{1 if resaltar is None or resaltar == k else .35};"></div></div>' for k, (v, t) in enumerate(zip(valores, textos)))
    ejes = "".join(f'<span style="flex:1;text-align:center;font-size:12px;font-weight:600;color:#3D4752;">{n}</span>' for n in NIVELES)
    return (f'<div style="height:{alto}px;display:flex;align-items:flex-end;gap:28px;padding:0 28px;border-bottom:1px solid #E3E7ED;">{cols}</div>'
            f'<div style="display:flex;gap:28px;padding:0 28px;">{ejes}</div>')


def kpi_html(titulo, valor, unidad, delta, bueno_si_sube, sufijo="%", vs="vs año anterior", color_titulo=None) -> str:
    if delta is None or not np.isfinite(delta):
        tendencia, color = "— <span style='color:#8A96A3;font-weight:400;'>sin año anterior</span>", "#8A96A3"
    else:
        bueno = (delta >= 0) == bueno_si_sube
        color = "#1E7A47" if bueno else "#B03A2E"
        tendencia = f"{'▲' if delta >= 0 else '▼'} {delta:+.1f}{sufijo} <span style='color:#8A96A3;font-weight:400;'>{vs}</span>"
    return f"""<div style="min-height:112px;height:100%;padding:14px 16px;background:#FFFFFF;border:1px solid #E3E7ED;border-radius:12px;display:flex;flex-direction:column;justify-content:space-between;box-sizing:border-box;font-family:'Segoe UI',Inter,sans-serif;">
      <span style="font-size:12px;font-weight:600;color:{GRIS};">{esc(titulo)}</span>
      <span><span style="font-size:26px;font-weight:700;color:{NAVY};">{esc(valor)}</span> <span style="font-size:13px;color:{GRIS};">{esc(unidad)}</span></span>
      <span style="font-size:12px;font-weight:600;color:{color};">{tendencia}</span></div>"""


def grid(celdas, columnas: int, gap: int = 14) -> str:
    return f'<div style="display:grid;grid-template-columns:repeat({columnas},minmax(0,1fr));gap:{gap}px;">{"".join(celdas)}</div>'


def grafica_linea(x, y, color, titulo, sub, formato):
    fig = go.Figure(go.Scatter(x=x, y=y, mode="lines+markers", line=dict(color=color, width=3), marker=dict(size=6),
                               hovertemplate="%{x}<br>%{y:" + formato + "}<extra></extra>"))
    fig.update_layout(height=300, margin=dict(l=12, r=12, t=64, b=10), plot_bgcolor="white", paper_bgcolor="white",
                      title=dict(text=f"<b>{titulo}</b><br><span style='font-size:12px;color:{GRIS}'>{sub}</span>",
                                 x=0.01, y=0.95, font=dict(size=15, color=TINTA)),
                      font=dict(family="Segoe UI, Inter, sans-serif", size=12), showlegend=False)
    fig.update_yaxes(gridcolor="#F0F2F5", zeroline=False, tickfont=dict(color="#8A96A3"))
    fig.update_xaxes(showgrid=False, tickfont=dict(color="#8A96A3"), tickangle=0, nticks=12)
    return fig


MAPA_INK, MAPA_MUTED, MAPA_FAINT, MAPA_LINE = TINTA, GRIS, "#8A96A3", "#E3E7ED"
MAPA_NIVEL = COL_NIVEL
MAPA_FONT = "Segoe UI, Inter, sans-serif"

CUADRANTES = [  # (nombre, acción sugerida, fondo, texto, posición) · X = cobertura, Y = demanda
    ("Alta rotación", "reabastecer", "rgba(46,158,91,.08)", "#1E6B41", "arriba-izq"),
    ("Exceso con salida", "transferir", "rgba(224,122,46,.08)", "#9A4A12", "arriba-der"),
    ("Equilibrado", "mantener", "rgba(138,150,163,.07)", "#5B6673", "abajo-izq"),
    ("Estancado", "reducir compras", "rgba(192,57,43,.07)", "#A32D21", "abajo-der"),
]


def simular(m: pd.DataFrame, dest_map: dict, aplicar: set, red: float, tras: float):
    """Cobertura esperada en 3 meses si se aplican las acciones marcadas (m: un renglón por producto–región del mes base).
    Reducir compras: la cobertura baja `red`.
    Transferir: cada origen intenta enviar `tras` de su inventario. Cada destino solo recibe lo que puede vender sin pasar
      a nivel Alto (cobertura ≤ límite Medio/Alto); lo que no cabe pasa a la siguiente región destino y, si no hay, se queda.
    Reabastecer: la cobertura sube al menos al límite Bajo/Medio.
    Regresa (cobertura con acciones, valor transferido, valor que no encontró destino)."""
    sin = m["meses_fut"].to_numpy()
    con = sin.copy()
    acc = m["accion"].to_numpy()
    enviado_val = sin_destino_val = 0.0
    if 0 in aplicar:
        con[acc == 0] *= 1 - red
    if 2 in aplicar:
        con[acc == 2] = np.maximum(con[acc == 2], CORTES[0])
    if 1 in aplicar:
        idx = {(p, r): i for i, (p, r) in enumerate(zip(m["product_id"], m["Región"]))}
        stock, vend = m["Stock_Restante_Mensual"].to_numpy(), m["units_sold"].to_numpy()
        precio = m["static_price"].to_numpy()
        tope = CORTES[1]                                     # un destino no debe pasar a Alto
        origenes = np.flatnonzero(acc == 1)
        origenes = origenes[np.argsort(-m["valor_inv"].to_numpy()[origenes])]   # primero los de mayor valor
        for i in origenes:
            quiere = stock[i] * tras
            if quiere <= 0:
                continue
            pendiente = quiere
            pid = m["product_id"].iat[i]
            for r in dest_map.get(pid, []):
                if pendiente <= 0:
                    break
                j = idx.get((pid, r))
                if j is None or j == i or vend[j] <= 0:
                    continue
                cabe = max(0.0, (tope - con[j]) * vend[j])      # unidades que el destino vende antes de llegar a Alto
                envio = min(pendiente, cabe)
                if envio > 0:
                    con[j] += envio / vend[j]
                    pendiente -= envio
            enviado = quiere - pendiente
            con[i] -= sin[i] * (enviado / stock[i]) if stock[i] else 0
            enviado_val += enviado * precio[i]
            sin_destino_val += pendiente * precio[i]
    return con, enviado_val, sin_destino_val


def mapa_cuadrante(pts: pd.DataFrame, corte_cob: float, corte_dem: float, mes_hoy: str, mes_fut: str,
                   etiquetas: bool, flechas: bool, con_acciones: bool, alto: int = 580):
    """pts: nombre, dem, c0 (cobertura hoy), c1 (en 3 meses sin acciones), c2 (en 3 meses final), valor, nivel2, detalle."""
    ly = np.log10(pts["dem"].clip(lower=1))
    y_min, y_max = min(ly.min(), np.log10(corte_dem)) - .15, max(ly.max(), np.log10(corte_dem)) + .15
    x_max = max(pts[["c0", "c1", "c2"]].max().max(), corte_cob) * 1.1 + (2 if etiquetas else 0)
    cy = np.log10(corte_dem)
    fig = go.Figure()
    zonas = {"arriba-izq": (0, corte_cob, cy, y_max), "arriba-der": (corte_cob, x_max, cy, y_max),
             "abajo-izq": (0, corte_cob, y_min, cy), "abajo-der": (corte_cob, x_max, y_min, cy)}
    for nom, acc, bg, fg, pos in CUADRANTES:
        x0, x1, y0, y1 = zonas[pos]
        fig.add_shape(type="rect", x0=x0, x1=x1, y0=y0, y1=y1, fillcolor=bg, line_width=0, layer="below")
        izq, arriba = pos.endswith("izq"), pos.startswith("arriba")
        fig.add_annotation(x=x0 if izq else x1, y=y1 if arriba else y0, xanchor="left" if izq else "right",
                           yanchor="top" if arriba else "bottom", align="left" if izq else "right", showarrow=False,
                           text=f"<b>{nom}</b><br><span style='font-size:11px'>{acc}</span>", font=dict(size=13, color=fg),
                           xshift=10 if izq else -10, yshift=-8 if arriba else 8)
    fig.add_shape(type="line", x0=corte_cob, x1=corte_cob, y0=y_min, y1=y_max, line=dict(color="#C5CDD6", width=1, dash="dot"))
    fig.add_shape(type="line", x0=0, x1=x_max, y0=cy, y1=cy, line=dict(color="#C5CDD6", width=1, dash="dot"))

    # Trayectoria hoy → 3 meses (horizontal: la demanda se mantiene en la del mes base)
    if flechas:
        for y, a, b in zip(ly, pts["c0"], pts["c2"]):
            if abs(b - a) > .05:
                fig.add_annotation(x=b, y=y, ax=a, ay=y, xref="x", yref="y", axref="x", ayref="y", showarrow=True, text="",
                                   arrowhead=2, arrowsize=1, arrowwidth=1.6, arrowcolor="rgba(91,102,115,.55)", standoff=7)
    else:
        xs, ys = [], []
        for y, a, b in zip(ly, pts["c0"], pts["c2"]):
            xs += [a, b, None]; ys += [y, y, None]
        fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color="rgba(91,102,115,.25)", width=1), hoverinfo="skip", showlegend=False))

    tam = 10 + 30 * np.sqrt(pts["valor"] / max(pts["valor"].max(), 1e-9))
    fig.add_trace(go.Scatter(x=pts["c0"], y=ly, mode="markers", name=f"Hoy ({mes_hoy})", text=pts["nombre"],
                             marker=dict(size=tam * .55, color="rgba(255,255,255,.9)", line=dict(color="#8A96A3", width=1.5)),
                             hovertemplate="<b>%{text}</b><br>Hoy: %{x:.1f} meses de cobertura<extra></extra>"))
    if con_acciones:
        cambia = (pts["c1"] - pts["c2"]).abs() > .05
        fig.add_trace(go.Scatter(x=pts.loc[cambia, "c1"], y=ly[cambia], mode="markers", name="Sin acciones", text=pts.loc[cambia, "nombre"],
                                 marker=dict(size=9, symbol="x-thin", color="#8A96A3", line=dict(color="#8A96A3", width=2)),
                                 hovertemplate="<b>%{text}</b><br>Sin acciones: %{x:.1f} meses<extra></extra>"))
    fig.add_trace(go.Scatter(x=pts["c2"], y=ly, mode="markers+text" if etiquetas else "markers", showlegend=False,
                             text=[f"{n}   " for n in pts["nombre"]], textposition="middle left",
                             textfont=dict(size=11, color=TINTA),
                             marker=dict(size=tam, color=[COL_NIVEL[k] for k in pts["nivel2"]], opacity=.88, line=dict(color="#FFFFFF", width=1.2)),
                             customdata=pts[["detalle"]].values, hovertemplate="%{customdata[0]}<extra></extra>"))
    for k, n in enumerate(NIVELES):
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name=f"{n} en {mes_fut}", marker=dict(size=11, color=COL_NIVEL[k])))

    ticks = [v for v in [10, 20, 50, 100, 200, 500, 1_000, 2_000, 5_000, 10_000, 20_000, 50_000, 100_000] if y_min <= np.log10(v) <= y_max]
    fig.update_layout(height=alto, margin=dict(l=10, r=10, t=10, b=10), plot_bgcolor="white", paper_bgcolor="white",
                      font=dict(family="Segoe UI, Inter, sans-serif", size=12, color=TINTA),
                      legend=dict(orientation="h", y=1.02, x=0, yanchor="bottom", font=dict(size=12)),
                      hoverlabel=dict(bgcolor="white", font_size=12, bordercolor="#E3E7ED"))
    fig.update_xaxes(range=[0, x_max], title=dict(text="Cobertura del inventario (meses)  →  más sobrestock", font=dict(size=12, color=GRIS)),
                     gridcolor="#F0F2F5", zeroline=False, tickfont=dict(color="#8A96A3"))
    fig.update_yaxes(range=[y_min, y_max], tickvals=[np.log10(v) for v in ticks], ticktext=[f"{v:,}" for v in ticks],
                     title=dict(text="Demanda esperada (uds/mes)", font=dict(size=12, color=GRIS)),
                     showgrid=False, zeroline=False, tickfont=dict(color="#8A96A3"))
    return fig


def ranking_movimiento(pts: pd.DataFrame, mes_hoy: str, mes_fut: str, con_acciones: bool):
    """Un renglón por producto–región: hoy → 3 meses sobre franjas de nivel. pts en el orden a mostrar (de arriba abajo)."""
    n = len(pts)
    y = np.arange(n)[::-1]                                   # el primero arriba
    x_max = max(pts[["c0", "c1", "c2"]].max().max(), CORTES[2]) * 1.08
    fig = go.Figure()
    bordes = [0, *CORTES, x_max]
    for k, nom in enumerate(NIVELES):                        # franjas de nivel
        fig.add_shape(type="rect", x0=bordes[k], x1=bordes[k + 1], y0=-.6, y1=n - .4, line_width=0, layer="below",
                      fillcolor=MAPA_NIVEL[k], opacity=.10 if k >= 2 else .07)
        fig.add_annotation(x=(bordes[k] + bordes[k + 1]) / 2, y=n - .4, yanchor="bottom", showarrow=False, yshift=4,
                           text=f"<b>{nom}</b>", font=dict(size=12, color=MAPA_NIVEL[k]))
    for i in range(n - 1):                                   # separadores de renglón
        fig.add_shape(type="line", x0=0, x1=x_max, y0=i + .5, y1=i + .5, line=dict(color="rgba(20,32,46,.05)", width=1), layer="below")

    xs, ys, xs_sin, ys_sin = [], [], [], []
    for yy, a, b, c in zip(y, pts["c0"], pts["c1"], pts["c2"]):
        xs += [a, c, None]; ys += [yy, yy, None]
        if con_acciones and abs(b - c) > .05:
            xs_sin += [a, b, None]; ys_sin += [yy, yy, None]
    if xs_sin:
        fig.add_trace(go.Scatter(x=xs_sin, y=ys_sin, mode="lines", line=dict(color="rgba(20,32,46,.18)", width=1, dash="dot"),
                                 hoverinfo="skip", showlegend=False))
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color="rgba(20,32,46,.35)", width=1.6), hoverinfo="skip", showlegend=False))
    fig.add_trace(go.Scatter(x=pts["c0"], y=y, mode="markers", name=f"Hoy · {mes_hoy}", text=pts["nombre"],
                             marker=dict(size=9, color="#FFFFFF", line=dict(color="#9EA1A6", width=1.3)),
                             hovertemplate="<b>%{text}</b><br>Hoy: %{x:.1f} meses<extra></extra>"))
    if con_acciones:
        cambia = (pts["c1"] - pts["c2"]).abs() > .05
        fig.add_trace(go.Scatter(x=pts.loc[cambia, "c1"], y=y[cambia.to_numpy()], mode="markers", name="Sin acciones", text=pts.loc[cambia, "nombre"],
                                 marker=dict(size=10, symbol="line-ns", color="#9EA1A6", line=dict(color="#9EA1A6", width=1.5)),
                                 hovertemplate="<b>%{text}</b><br>Sin acciones: %{x:.1f} meses<extra></extra>"))
    fig.add_trace(go.Scatter(x=pts["c2"], y=y, mode="markers", showlegend=False,
                             marker=dict(size=13, color=[MAPA_NIVEL[k] for k in pts["nivel2"]], line=dict(color="#FFFFFF", width=1.5)),
                             customdata=pts[["detalle"]].values, hovertemplate="%{customdata[0]}<extra></extra>"))
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name=f"{mes_fut} · color según nivel",
                             marker=dict(size=10, color=MAPA_NIVEL[1], line=dict(color="#FFFFFF", width=1))))

    # Columnas a la derecha: cambio en meses y valor del inventario
    for yy, a, c, v in zip(y, pts["c0"], pts["c2"], pts["valor"]):
        d = c - a
        color = "#1E7A47" if d < -.05 else ("#B03A2E" if d > .05 else MAPA_FAINT)
        fig.add_annotation(x=1, xref="paper", y=yy, xanchor="left", showarrow=False, xshift=14, align="left",
                           text=f"<span style='color:{color}'>{d:+.1f} m</span>", font=dict(size=11.5))
        fig.add_annotation(x=1, xref="paper", y=yy, xanchor="left", showarrow=False, xshift=92, align="left",
                           text=dinero(v), font=dict(size=11.5, color=MAPA_MUTED))
    for txt, sh in (("Δ MESES", 14), ("INVENTARIO", 92)):
        fig.add_annotation(x=1, xref="paper", y=n - .4, yanchor="bottom", xanchor="left", showarrow=False, xshift=sh, yshift=4,
                           text=f"<b>{txt.capitalize()}</b>", font=dict(size=11, color=MAPA_FAINT))

    etiquetas = [f"<b>{esc(p_)}</b>  <span style='color:{MAPA_FAINT}'>{esc(r_)}</span>" for p_, r_ in zip(pts["producto"], pts["region"])]
    fig.update_layout(height=110 + 34 * n, margin=dict(l=4, r=170, t=30, b=40), plot_bgcolor="white", paper_bgcolor="rgba(0,0,0,0)", showlegend=False,
                      font=dict(family=MAPA_FONT, size=12, color=MAPA_INK),
                      legend=dict(orientation="h", y=1, x=0, yanchor="top", yref="container", font=dict(size=11, color=MAPA_MUTED),
                                  bgcolor="rgba(0,0,0,0)"),
                      hoverlabel=dict(bgcolor="white", font_size=12, bordercolor="#E3E7ED"))
    fig.update_yaxes(range=[-.6, n - .4], tickvals=y, ticktext=etiquetas, showgrid=False, zeroline=False, ticks="",
                     tickfont=dict(size=12, color=MAPA_INK), automargin=True)
    fig.update_xaxes(range=[0, x_max], showgrid=False, zeroline=False, showline=True, linecolor="#E3E7ED", ticks="outside",
                     ticklen=4, tickcolor="#E3E7ED", tickfont=dict(size=11, color=MAPA_FAINT),
                     title=dict(text="Cobertura del inventario (meses)  →  más sobrestock", font=dict(size=12, color=GRIS)))
    return fig


# ──────────────────────────────────────────────────────────────────────────────
# Encabezado, filtros y carga
# ──────────────────────────────────────────────────────────────────────────────
if not ARCHIVO.exists():
    st.error(f"No se encontró {ARCHIVO.name}. Colócalo en la misma carpeta que app.py.")
    st.stop()

logo_danu = ""
if LOGO_DANU.exists():
    b64 = base64.b64encode(LOGO_DANU.read_bytes()).decode()
    logo_danu = (f'<span style="width:1px;height:36px;background:rgba(255,255,255,.22);"></span>'
                 f'<img src="data:image/png;base64,{b64}" alt="danu analítica" style="height:34px;">')

st.html(f"""
<div style="padding:16px 22px;border-radius:12px;background:{NAVY};color:#FFFFFF;display:flex;align-items:center;justify-content:space-between;font-family:'Segoe UI',Inter,sans-serif;">
  <div style="display:flex;align-items:center;gap:18px;">
    <div style="display:flex;align-items:center;gap:10px;">
      {svg_img(LOGO_NEXXUS, 30, 30)}
      <div style="display:flex;flex-direction:column;line-height:1;"><span style="font-size:17px;font-weight:700;">nexxus</span>
        <span style="font-size:8px;font-weight:600;letter-spacing:.32em;color:#A9C4E4;margin-top:3px;">CONSULTING</span></div>
    </div>
    <span style="width:1px;height:36px;background:rgba(255,255,255,.22);"></span>
    <div style="display:flex;flex-direction:column;gap:3px;">
      <span style="font-size:10px;font-weight:600;letter-spacing:.12em;text-transform:uppercase;color:#A9C4E4;">Tablero de inventario</span>
      <span style="font-size:22px;font-weight:700;">Movimiento de Inventario - DanuStore</span>
    </div>
  </div>
  <div style="display:flex;align-items:center;gap:18px;">{logo_danu}</div>
</div>""")

datos = cargar_datos(ARCHIVO)
with st.spinner("Preparando el modelo de pronóstico…"):
    dp, CORTES, CORTES_DEM = entrenar_y_pronosticar(datos)

REGIONES = sorted(dp["Región"].unique())


def orden_producto(nombre: str):
    sub, num = nombre.split(" · P")
    return sub, int(num)


def limpiar_filtros():
    for k in ("f_reg", "f_prod", "f_cat", "f_sub"):
        st.session_state[k] = "Todas" if k != "f_prod" else "Todos"
    st.session_state["f_fechas"] = (date(2024, 1, 1), date(2025, 12, 31))


with st.container(border=True):
    c1, c2, c3, c4, c5, c6 = st.columns([1, 1, 1, 1, 2, 0.55], vertical_alignment="bottom")
    f_reg = c1.selectbox("Región", ["Todas"] + REGIONES, key="f_reg")
    f_cat = c3.selectbox("Categoría", ["Todas"] + ORDEN_CATEGORIA, key="f_cat")
    base_sub = dp if f_cat == "Todas" else dp[dp["Categoría"] == f_cat]
    f_sub = c4.selectbox("Subcategoría", ["Todas"] + sorted(base_sub["Subcategoría"].unique()), key="f_sub")
    base_prod = base_sub if f_sub == "Todas" else base_sub[base_sub["Subcategoría"] == f_sub]
    f_prod = c2.selectbox("Producto", ["Todos"] + sorted(base_prod["Producto"].unique(), key=orden_producto), key="f_prod")
    fechas = c5.date_input("Rango de fechas", value=(date(2024, 1, 1), date(2025, 12, 31)), min_value=date(2024, 1, 1),
                           max_value=date(2025, 12, 31), format="DD/MM/YYYY", key="f_fechas")
    c6.button("Limpiar", on_click=limpiar_filtros, width="stretch")

ini, fin = fechas if isinstance(fechas, tuple) and len(fechas) == 2 else (date(2024, 1, 1), date(2025, 12, 31))
T0, T1 = (ini.year - 2024) * 12 + ini.month - 1, (fin.year - 2024) * 12 + fin.month - 1

mask = pd.Series(True, index=dp.index)
if f_reg != "Todas": mask &= dp["Región"] == f_reg
if f_cat != "Todas": mask &= dp["Categoría"] == f_cat
if f_sub != "Todas": mask &= dp["Subcategoría"] == f_sub
if f_prod != "Todos": mask &= dp["Producto"] == f_prod
dims = dp[mask]                                         # filtros de dimensión (sin fecha)
PRODUCTOS_OK = sorted(dims["Producto"].unique(), key=orden_producto)
REGIONES_OK = sorted(dims["Región"].unique())
MESES_OK = list(range(T0, T1 + 1))


@st.cache_data
def destinos(_d: pd.DataFrame, t: int) -> dict:
    """Destino de transferencia: misma pieza en otra región con nivel Bajo pronosticado,
    priorizando la que necesita reabastecer y luego la de menor cobertura."""
    m = _d[(_d["t"] == t) & (_d["nivel_pred"] == 0)].copy()
    m["prio"] = (m["accion"] != 2).astype(int)
    m = m.sort_values(["product_id", "prio", "Dias_Cobertura_Inventario"])
    return {pid: list(g["Región"]) for pid, g in m.groupby("product_id")}


def texto_accion(fila, dest_map) -> str:
    if fila["accion"] != 1:
        return ACCIONES[fila["accion"]]
    opciones = [r for r in dest_map.get(fila["product_id"], []) if r != fila["Región"]]
    return f"Transferir a {opciones[0]}" if opciones else "Liquidar (sin región destino)"


tab_plan, tab_diag = st.tabs(["Plan de acción", "Diagnóstico"])

# ──────────────────────────────────────────────────────────────────────────────
# Página 1 · Plan de acción (predictivo)
# ──────────────────────────────────────────────────────────────────────────────
with tab_plan:
    plan = dims[dims["t"] == T1].copy()                  # un renglón por producto–región del mes base
    dest_map = destinos(dp, T1)
    plan["Acción"] = [texto_accion(f, dest_map) for _, f in plan.iterrows()]

    encabezado_seccion("01", "check", "Acciones recomendadas",
                       "Acción sugerida por producto–región según el nivel de sobrestock pronosticado")
    tarjetas = []
    for k, nombre in enumerate(ACCIONES):
        g = plan[plan["accion"] == k]
        top = ", ".join(g["Región"].value_counts().head(2).index) or "—"
        niveles = "".join(pill(l, chico=True) for l in NIVELES_ACCION[k])
        tarjetas.append(f"""
        <div style="min-height:218px;height:100%;padding:16px 18px;background:#FFFFFF;border:1px solid #E3E7ED;border-top:4px solid {COL_ACCION[k]};border-radius:12px;display:flex;flex-direction:column;justify-content:space-between;box-sizing:border-box;font-family:'Segoe UI',Inter,sans-serif;color:{TINTA};">
          <span style="display:flex;align-items:center;justify-content:space-between;gap:8px;"><span style="display:flex;align-items:center;gap:8px;font-size:14px;font-weight:700;">{punto(COL_ACCION[k], 11)}{nombre}</span><span style="display:flex;gap:4px;">{niveles}</span></span>
          <span><span style="font-size:30px;font-weight:700;color:{NAVY};">{len(g):,}</span> <span style="font-size:12px;color:{GRIS};">producto–región</span></span>
          <span style="font-size:12px;color:{GRIS};"><b style="color:{TINTA};">{dinero(g['valor_inv'].sum())}</b> valor del inventario</span>
          <span style="display:flex;flex-direction:column;gap:4px;border-top:1px solid #EEF1F4;padding-top:10px;font-size:12px;">
            <span>{QUE_HACER[k]}</span><span style="color:{GRIS};">Regiones principales: <b style="color:{TINTA};">{esc(top)}</b></span></span>
        </div>""")
    st.html(grid(tarjetas, 4))

    with st.container(border=True):
        a, b = st.columns([3, 1], vertical_alignment="center")
        ver = a.segmented_control("Lista de prioridades", ["Prioridades"] + ACCIONES, default="Prioridades", key="plan_lista")
        todas = b.toggle("Ver todas", key="plan_todas")
        ver = ver or "Prioridades"
        if ver == "Prioridades":
            lista = plan[plan["accion"] <= 1].sort_values("valor_inv", ascending=False)
            sub = "Reducir compras y transferir, ordenada por valor del inventario"
        elif ver == "Reabastecer":
            lista = plan[plan["accion"] == 2].sort_values("Dias_Cobertura_Inventario")
            sub = "Ordenada de menor a mayor cobertura de inventario"
        else:
            lista = plan[plan["accion"] == ACCIONES.index(ver)].sort_values("valor_inv", ascending=False)
            sub = "Ordenada por valor del inventario"
        filas_vista = lista if todas else lista.head(12)
        filas = [[f'<span style="color:#8A96A3;">{i}</span>', f"<b>{esc(f['Producto'])}</b>", esc(f["Región"]), pill(int(f["nivel_pred"])),
                  f'<span style="display:inline-flex;align-items:center;gap:7px;font-weight:600;">{punto(COL_ACCION[f["accion"]])}{esc(f["Acción"])}</span>',
                  f"<b>{dinero(f['valor_inv'])}</b>", meses(f["Dias_Cobertura_Inventario"])]
                 for i, (_, f) in enumerate(filas_vista.iterrows(), 1)]
        st.html(f'<div style="font-family:Segoe UI,Inter,sans-serif;font-size:12px;color:{GRIS};margin-bottom:8px;">{sub} · {len(lista):,} producto–región</div>' +
                tabla([("#", "l"), ("Producto", "l"), ("Región", "l"), ("Sobrestock", "l"), ("Acción recomendada", "l"),
                       ("Valor del inventario", "r"), ("Cobertura", "r")], filas, alto_max=560 if todas else None, fuente=13))

    # 02 · Mapa de cuadrantes: hoy → 3 meses, con simulador de acciones
    encabezado_seccion("02", "compass", "¿Hacia dónde va el inventario en 3 meses?",
                       "Posición de hoy y posición esperada por el modelo; marca acciones para ver cómo cambiaría el escenario")
    mes_hoy, mes_fut = etiqueta_mes(T1), etiqueta_mes(T1 + 3)
    with st.container(border=True):
        a, b, c = st.columns([2.2, 1.3, .7], vertical_alignment="bottom")
        vista = a.segmented_control("Ver por", ["Región", "Subcategoría", "Producto–región"], default="Región", key="mapa_vista")
        vista = vista or "Región"
        es_rank = vista == "Producto–región"
        if es_rank:
            orden_r = b.selectbox("Ordenar por", ["Valor del inventario", "Mayor mejora", "Mayor deterioro", "Impacto de las acciones"],
                                  key="rank_orden")
            mas = c.toggle("Ver 40", key="rank_mas", help="Por defecto se muestran 15 producto–región")
        st.html(f'<div style="font-family:Segoe UI,Inter,sans-serif;font-size:14px;font-weight:700;color:{TINTA};margin-top:6px;">'
                f'Simula las acciones recomendadas <span style="font-size:12px;font-weight:400;color:{GRIS};">· marca una o varias y ajusta qué tan fuerte aplicarlas</span></div>')

        def cuenta(k):
            g = plan[plan["accion"] == k]
            return (f'<div style="font-family:Segoe UI,Inter,sans-serif;font-size:12px;color:{GRIS};margin:-4px 0 2px;">'
                    f'Aplica a <b style="color:{TINTA};">{len(g):,}</b> producto–región · <b style="color:{TINTA};">{dinero(g["valor_inv"].sum())}</b> de inventario</div>')

        c1, c2, c3 = st.columns(3)
        with c1.container(border=True):
            ap_red = st.checkbox(":red[●] **Reducir compras**", key="sim_red")
            st.caption("Para productos con exceso y poca demanda: se compra menos el próximo pedido, así baja el inventario.")
            st.html(cuenta(0))
            red = st.slider("¿Cuánto comprar menos?", 0, 80, 30, 10, format="%d%%", key="sim_red_pct", disabled=not ap_red,
                            help="El inventario esperado en 3 meses baja este porcentaje.") / 100
        with c2.container(border=True):
            ap_tra = st.checkbox(":orange[●] **Transferir**", key="sim_tra")
            st.caption("Para productos en Crítico con demanda: se manda inventario a regiones donde sí se vende.")
            st.html(cuenta(1))
            tras = st.slider("¿Qué parte del inventario mover?", 0, 100, 50, 10, format="%d%%", key="sim_tra_pct", disabled=not ap_tra,
                             help=f"Cada región destino solo recibe lo que puede vender sin pasar de {CORTES[1]:.1f} meses de cobertura (nivel Alto). "
                                  "Lo que no cabe va a otra región o se queda en el origen.") / 100
        with c3.container(border=True):
            ap_rea = st.checkbox(":green[●] **Reabastecer**", key="sim_rea")
            st.caption("Para productos con poco inventario y mucha demanda: se compra para no quedarse sin piezas.")
            st.html(cuenta(2))
            st.html(f'<div style="font-family:Segoe UI,Inter,sans-serif;font-size:12px;color:{GRIS};padding:10px 12px;border-radius:8px;background:#F4F6F9;margin-top:6px;">'
                    f'Se compra lo necesario para tener al menos <b style="color:{TINTA};">{CORTES[0]:.1f} meses</b> de inventario. No requiere ajuste.</div>')
        aplicar = {k for k, v in [(0, ap_red), (1, ap_tra), (2, ap_rea)] if v}

        todo_t = dp[dp["t"] == T1].copy()                  # todas las regiones: las transferencias pueden salir del filtro
        todo_t["meses_con"], val_trans, val_sin_dest = simular(todo_t, dest_map, aplicar, red, tras)
        base = plan.join(todo_t["meses_con"])
        cortes_niv = [-np.inf, *CORTES, np.inf]
        base["nivel_con"] = pd.cut(base["meses_con"], cortes_niv, labels=False).astype(int)
        base["nivel_sin"] = base["nivel_pred"].astype(int)

        if es_rank:
            orden_col = {"Valor del inventario": (base["valor_inv"], False), "Mayor mejora": (base["meses_con"] - base["meses_inv"], True),
                         "Mayor deterioro": (base["meses_con"] - base["meses_inv"], False),
                         "Impacto de las acciones": (base["meses_con"] - base["meses_fut"], True)}[orden_r]
            sel = base.assign(_o=orden_col[0]).sort_values("_o", ascending=orden_col[1]).head(40 if mas else 15)
            pts = pd.DataFrame({"nombre": sel["Producto"] + " · " + sel["Región"], "producto": sel["Producto"], "region": sel["Región"],
                                "dem": sel["units_expected"],
                                "c0": sel["meses_inv"], "c1": sel["meses_fut"], "c2": sel["meses_con"], "valor": sel["valor_inv"],
                                "nivel0": sel["nivel_obs"].astype(int), "nivel1": sel["nivel_sin"], "nivel2": sel["nivel_con"]})
        else:
            col = "Subcategoría" if vista == "Subcategoría" else "Región"
            w = base.assign(p0=base["meses_inv"] * base["valor_inv"], p1=base["meses_fut"] * base["valor_inv"], p2=base["meses_con"] * base["valor_inv"])
            g = w.groupby(col).agg(dem=("units_expected", "mean"), p0=("p0", "sum"), p1=("p1", "sum"), p2=("p2", "sum"),
                                   valor=("valor_inv", "sum")).reset_index()
            g = g[g["valor"] > 0]
            pts = pd.DataFrame({"nombre": g[col], "dem": g["dem"], "c0": g["p0"] / g["valor"], "c1": g["p1"] / g["valor"],
                                "c2": g["p2"] / g["valor"], "valor": g["valor"]})
            for k in range(3):
                pts[f"nivel{k}"] = pd.cut(pts[f"c{k}"], cortes_niv, labels=False).astype(int)
        con_acc = bool(aplicar)
        pts["detalle"] = [f"<b>{esc(n)}</b><br>Cobertura hoy: {a_:.1f} meses ({NIVELES[k0]})<br>"
                          f"En {mes_fut} sin acciones: {b_:.1f} meses ({NIVELES[k1]})<br>" +
                          (f"<b>Con acciones: {c_:.1f} meses ({NIVELES[k2]})</b><br>" if con_acc else "") +
                          f"Demanda esperada: {d_:,.0f} uds/mes<br>Inventario: {dinero(v)}"
                          for n, a_, b_, c_, k0, k1, k2, d_, v in zip(pts["nombre"], pts["c0"], pts["c1"], pts["c2"], pts["nivel0"],
                                                                     pts["nivel1"], pts["nivel2"], pts["dem"], pts["valor"])]

        if pts.empty:
            st.info("No hay datos para los filtros seleccionados.")
        else:
            izq, der = st.columns([2.3, 1])
            with izq:
                if es_rank:
                    item = lambda simb, txt: f'<span style="display:inline-flex;align-items:center;gap:7px;">{simb}{txt}</span>'
                    st.html(f'<div style="font-family:{MAPA_FONT};display:flex;gap:26px;font-size:11.5px;color:{MAPA_MUTED};margin:4px 0 6px;">' +
                            item('<span style="width:9px;height:9px;border-radius:50%;border:1.3px solid #9EA1A6;"></span>', f"Hoy · {mes_hoy}") +
                            item(f'<span style="width:11px;height:11px;border-radius:50%;background:{MAPA_NIVEL[1]};"></span>', f"{mes_fut} · color según nivel") +
                            (item('<span style="width:1.5px;height:12px;background:#9EA1A6;"></span>', "Sin acciones") if con_acc else "") + "</div>")
                fig = (ranking_movimiento(pts, mes_hoy, mes_fut, con_acc) if es_rank else
                       mapa_cuadrante(pts, CORTES[1], CORTES_DEM[0], mes_hoy, mes_fut, etiquetas=True, flechas=True, con_acciones=con_acc))
                st.plotly_chart(fig, width="stretch", key="mapa_fig", config={"displayModeBar": False})

            ac_sin, ac_con = base["nivel_sin"] >= 2, base["nivel_con"] >= 2
            v = base["valor_inv"]
            cob = lambda col_: (base[col_] * v).sum() / v.sum() if v.sum() else 0
            nivel_fin = base["nivel_con"]

            def fila_cmp(txt, color, sin_, con_, fmt):
                if con_acc:
                    c = "#1E7A47" if con_ < sin_ - 1e-9 else ("#B03A2E" if con_ > sin_ + 1e-9 else TINTA)
                    valor = (f'<span style="font-size:12px;color:#8A96A3;text-decoration:line-through;">{fmt(sin_)}</span> '
                             f'<b style="font-size:19px;color:{c};">{fmt(con_)}</b>')
                else:
                    valor = f'<b style="font-size:19px;color:{NAVY};">{fmt(sin_)}</b>'
                return (f'<div style="display:flex;align-items:baseline;justify-content:space-between;gap:8px;padding:10px 0;border-bottom:1px solid #EEF1F4;">'
                        f'<span style="display:flex;align-items:center;gap:8px;font-size:13px;">{punto(color)}{txt}</span><span>{valor}</span></div>')

            n_ = lambda x: f"{int(x):,}"
            m_ = lambda x: f"{x:.1f} m"
            sube, baja = int((nivel_fin > base["nivel_obs"]).sum()), int((nivel_fin < base["nivel_obs"]).sum())
            filas_p = ('<div>' +
                       fila_cmp("En Alto o Crítico", COL_NIVEL[3], ac_sin.sum(), ac_con.sum(), n_) +
                       fila_cmp("Valor en riesgo", COL_NIVEL[2], v[ac_sin].sum(), v[ac_con].sum(), dinero) +
                       fila_cmp("Cobertura promedio", NAVY, cob("meses_fut"), cob("meses_con"), m_) + '</div>')
            mov = (f'<div style="font-size:12px;color:{GRIS};">vs hoy: <b style="color:#B03A2E;">▲ {sube:,}</b> suben · '
                   f'<b style="color:#1E7A47;">▼ {baja:,}</b> bajan · <b style="color:{TINTA};">{len(base) - sube - baja:,}</b> igual</div>')
            if 1 in aplicar:
                mov += (f'<div style="font-size:12px;color:{GRIS};line-height:1.6;">Transferido: <b style="color:{TINTA};">{dinero(val_trans)}</b>'
                        f' · sin destino con capacidad: <b style="color:#B03A2E;">{dinero(val_sin_dest)}</b></div>')
            nota = ("Tachado: sin acciones · en color: con las acciones marcadas." if con_acc
                    else "Marca una o más acciones para comparar contra no hacer nada.")
            der.html(tarjeta(f"Escenario {mes_fut}", f"{len(base):,} producto–región con los filtros actuales",
                             filas_p + mov + f'<span style="font-size:11px;color:{GRIS};">{nota}</span>', alto=560))

        st.html(f'<div style="font-family:{MAPA_FONT};font-size:11px;color:{GRIS};line-height:1.6;">' +
                (f'Cada renglón va de hoy ({mes_hoy}, círculo hueco) a {mes_fut} (círculo de color por nivel); las franjas de fondo son los niveles de sobrestock. '
                 if es_rank else
                 f'Círculo hueco: hoy ({mes_hoy}). Círculo de color: posición esperada en {mes_fut}, por nivel de sobrestock; el tamaño es el valor del inventario. ') +
                f'Sin acciones, la cobertura futura pondera la cobertura típica de cada nivel por la probabilidad de la red neuronal; con acciones se aplican los '
                f'supuestos del escenario y la marca gris indica dónde quedaría sin intervenir. La demanda se mantiene en la del mes base. '
                f'Cortes: riesgo &gt; {CORTES[1]:.1f} meses de cobertura · demanda baja &lt; {CORTES_DEM[0]:,.0f} uds/mes.</div>')

    encabezado_seccion("03", "pin", "Distribución regional", "Inventario por liberar y transferencias recomendadas entre regiones")
    izq, der = st.columns(2)
    liberar = plan[plan["accion"] <= 1].groupby("Región")["valor_inv"].sum().reindex(REGIONES_OK, fill_value=0).sort_values(ascending=False)
    izq.html(tarjeta("Inventario por liberar por región", "Valor del inventario con acción de reducir compras o transferir",
                     barras_h([(r, v, dinero(v)) for r, v in liberar.items()]), alto=420))
    trans = plan[plan["accion"] == 1].sort_values("valor_inv", ascending=False)
    filas_t = []
    for _, f in trans.iterrows():
        destino = f["Acción"].replace("Transferir a ", "") if f["Acción"].startswith("Transferir a") else "Sin destino: liquidar"
        cob_dest = dp[(dp["t"] == T1) & (dp["product_id"] == f["product_id"]) & (dp["Región"] == destino)]["Dias_Cobertura_Inventario"]
        cob = f"{f['meses_inv']:.1f} → {cob_dest.iloc[0] / 30.4:.1f} meses" if len(cob_dest) else meses(f["Dias_Cobertura_Inventario"])
        filas_t.append([f"<b>{esc(f['Producto'])}</b>", esc(f["Región"]), f'<b style="color:#1E6B41;">{esc(destino)}</b>', dinero(f["valor_inv"]), cob])
    der.html(tarjeta("Transferencias recomendadas", f"Producto–región en nivel Crítico y región destino con mayor rotación · {len(trans)} casos",
                     tabla([("Producto", "l"), ("Origen", "l"), ("Destino", "l"), ("Inventario", "r"), ("Cobertura (origen → destino)", "r")],
                           filas_t, alto_max=300), alto=420))

    encabezado_seccion("04", "search", "Consulta de pronósticos", "Pronóstico del modelo por producto o por región para el mes base seleccionado")
    if not PRODUCTOS_OK:
        st.info("No hay productos para los filtros seleccionados.")
    else:
        st.markdown("##### Consulta por producto")
        a, b, c = st.columns(3)
        prod_def = "Botas · P113" if "Botas · P113" in PRODUCTOS_OK else PRODUCTOS_OK[0]
        q_prod = a.selectbox("Producto", PRODUCTOS_OK, index=PRODUCTOS_OK.index(prod_def), key="q_prod")
        q_reg = b.selectbox("Región", REGIONES_OK, index=REGIONES_OK.index("Sureste") if "Sureste" in REGIONES_OK else 0, key="q_reg")
        q_mes = c.selectbox("Mes base", MESES_OK, index=len(MESES_OK) - 1, format_func=etiqueta_mes, key="q_mes")
        fila = dp[(dp["Producto"] == q_prod) & (dp["Región"] == q_reg) & (dp["t"] == q_mes)].iloc[0]
        k, acc = int(fila["nivel_pred"]), int(fila["accion"])
        texto = texto_accion(fila, destinos(dp, q_mes))
        real = fila["nivel_real"]
        porque = [f"Cobertura del inventario en el mes base: {meses(fila['Dias_Cobertura_Inventario'])} al ritmo de venta vigente.",
                  f"Demanda esperada {DEMANDA[int(fila['demanda'])].lower()}: {int(fila['units_expected']):,} unidades mensuales.",
                  f"Nivel observado en el trimestre pronosticado: {NIVELES[int(real)]}." if pd.notna(real)
                  else "El trimestre pronosticado aún no tiene datos observados."]
        cuerpo = (f'<div style="display:flex;align-items:center;gap:12px;"><span style="font-size:13px;color:{GRIS};">Sobrestock pronosticado</span>{pill(k)}</div>'
                  f'<div style="display:flex;align-items:center;gap:12px;padding:12px 14px;border-radius:10px;background:#F4F6F9;border-left:4px solid {COL_ACCION[acc]};">'
                  f'{punto(COL_ACCION[acc], 12)}<span style="display:flex;flex-direction:column;gap:2px;"><span style="font-size:16px;font-weight:700;">{esc(texto)}</span>'
                  f'<span style="font-size:12px;color:#3D4752;">{QUE_HACER[acc]}</span></span></div>'
                  '<ul style="margin:0;padding-left:18px;font-size:13px;color:#3D4752;display:flex;flex-direction:column;gap:6px;">' +
                  "".join(f"<li>{esc(w)}</li>" for w in porque) + "</ul>")
        izq, der = st.columns(2)
        izq.html(tarjeta("Resultado del pronóstico", f"{q_prod} · {q_reg}", cuerpo, alto=320))
        probs = [fila[f"p_{n}"] for n in NIVELES]
        der.html(tarjeta("Probabilidad por nivel", "Distribución de probabilidad estimada por el modelo",
                         barras_nivel(probs, [f"{p:.0%}" for p in probs], alto=200, resaltar=k), alto=320))

        st.markdown("##### Consulta por región")
        a, b = st.columns(2)
        r_reg = a.selectbox("Región", REGIONES_OK, index=REGIONES_OK.index("Noroeste") if "Noroeste" in REGIONES_OK else 0, key="r_reg")
        r_mes = b.selectbox("Mes base", MESES_OK, index=len(MESES_OK) - 1, format_func=etiqueta_mes, key="r_mes")
        reg = dims[(dims["Región"] == r_reg) & (dims["t"] == r_mes)].sort_values(["nivel_pred", "Dias_Cobertura_Inventario"], ascending=False)
        dm = destinos(dp, r_mes)
        conteo = [int((reg["nivel_pred"] == i).sum()) for i in range(4)]
        izq, der = st.columns(2)
        izq.html(tarjeta("Productos por nivel de sobrestock", f"Número de productos por nivel pronosticado en {r_reg}",
                         barras_nivel(conteo, conteo, alto=250), alto=400))
        filas_r = [[f"<b>{esc(f['Producto'])}</b>", pill(int(f["nivel_pred"])),
                    f'<span style="display:inline-flex;align-items:center;gap:7px;">{punto(COL_ACCION[f["accion"]])}{esc(texto_accion(f, dm))}</span>']
                   for _, f in reg.iterrows()]
        der.html(tarjeta("Productos de la región", "Ordenados de mayor a menor nivel de sobrestock",
                         tabla([("Producto", "l"), ("Sobrestock", "l"), ("Acción recomendada", "l")], filas_r, alto_max=290), alto=400))

# ──────────────────────────────────────────────────────────────────────────────
# Página 2 · Diagnóstico (descriptivo)
# ──────────────────────────────────────────────────────────────────────────────
with tab_diag:
    rango = dims[dims["t"].between(T0, T1)]              # flujos: se acumulan en el rango
    cierre = dims[dims["t"] == T1]                       # inventario: solo el último mes del rango
    if rango.empty:
        st.warning("No hay datos para los filtros seleccionados.")
    else:
        cur_t = [t for t in MESES_OK if t >= 12]
        cur, prev = dims[dims["t"].isin(cur_t)], dims[dims["t"].isin([t - 12 for t in cur_t])]

        def var(a, b):
            return (a / b - 1) * 100 if cur_t and b else None

        cierre_prev = dims[dims["t"] == T1 - 12] if T1 >= 12 else dims.iloc[0:0]
        alto = lambda x: x.loc[x["nivel_obs"] >= 2, "valor_inv"].sum() / x["valor_inv"].sum() * 100 if x["valor_inv"].sum() else 0
        ventas, uds, esp = rango["sales_amount"].sum(), rango["units_sold"].sum(), rango["units_expected"].sum()
        cumpl = lambda x: x["units_sold"].sum() / x["units_expected"].sum() if x["units_expected"].sum() else np.nan
        inv = cierre["valor_inv"].sum()
        div_u, suf_u = (1e6, "M uds") if uds >= 1e6 else (1e3, "K uds") if uds >= 1e3 else (1, "uds")
        st.html(grid([
            kpi_html("Ventas", dinero(ventas).replace(" M", "").replace(" K", ""), "M" if ventas >= 1e6 else "K",
                     var(cur["sales_amount"].sum(), prev["sales_amount"].sum()), True),
            kpi_html("Unidades vendidas", f"{uds / div_u:,.2f}" if div_u > 1 else f"{uds:,}", suf_u,
                     var(cur["units_sold"].sum(), prev["units_sold"].sum()), True),
            kpi_html("Cumplimiento de venta", f"{uds / esp * 100:.1f}", "%",
                     var(cumpl(cur), cumpl(prev)) if cur_t else None, True),
            kpi_html("Valor del inventario al cierre", dinero(inv).replace(" M", "").replace(" K", ""), "M" if inv >= 1e6 else "K",
                     (inv / cierre_prev["valor_inv"].sum() - 1) * 100 if len(cierre_prev) and cierre_prev["valor_inv"].sum() else None, False),
            kpi_html("Inventario en Alto y Crítico", f"{alto(cierre):.1f}", "% del valor",
                     alto(cierre) - alto(cierre_prev) if len(cierre_prev) else None, False, sufijo=" pp"),
        ], 5))

        # 01 · Lista prioritaria
        encabezado_seccion("01", "alert", "Producto–región con mayor sobrestock", "Lista priorizada por cobertura del inventario al cierre del periodo")
        with st.container(border=True):
            todas_d = st.toggle("Ver todas", key="diag_todas")
            peor = cierre.sort_values("Dias_Cobertura_Inventario", ascending=False)
            peor_v = peor if todas_d else peor.head(12)
            ant = dp.set_index(["region", "product_id", "t"])["Dias_Cobertura_Inventario"]
            filas = []
            for i, (_, f) in enumerate(peor_v.iterrows(), 1):
                previo = ant.get((f["region"], f["product_id"], T1 - 1)) if T1 > 0 else None
                if previo is None:
                    var_txt = '<span style="color:#8A96A3;">—</span>'
                else:
                    dm_ = (f["Dias_Cobertura_Inventario"] - previo) / 30.4
                    var_txt = f'<span style="font-size:12px;font-weight:600;color:{"#B03A2E" if dm_ > 0 else "#1E7A47"};">{"▲ +" if dm_ >= 0 else "▼ "}{dm_:.1f} meses</span>'
                filas.append([f'<span style="color:#8A96A3;">{i}</span>', f"<b>{esc(f['Producto'])}</b>", esc(f["Región"]), pill(int(f["nivel_obs"])),
                              f"<b>{meses(f['Dias_Cobertura_Inventario'])}</b>", var_txt, dinero(f["valor_inv"]), pct(f["units_sold"], f["units_expected"])])
            st.html(f'<div style="font-family:Segoe UI,Inter,sans-serif;font-size:12px;color:{GRIS};margin-bottom:8px;">Ordenada de mayor a menor cobertura; la variación compara contra el mes anterior · {len(peor):,} producto–región</div>' +
                    tabla([("#", "l"), ("Producto", "l"), ("Región", "l"), ("Sobrestock", "l"), ("Cobertura", "r"), ("Variación", "r"),
                           ("Valor del inventario", "r"), ("Cumplimiento de venta", "r")], filas, alto_max=560 if todas_d else None, fuente=13))

        # 02 · Inventario por nivel
        encabezado_seccion("02", "layers", "Inventario por nivel de sobrestock",
                           "Composición del inventario al cierre del periodo según los meses que tardaría en venderse")
        rangos = [f"≤ {CORTES[0]:.1f} meses", f"{CORTES[0]:.1f}–{CORTES[1]:.1f} meses", f"{CORTES[1]:.1f}–{CORTES[2]:.1f} meses", f"> {CORTES[2]:.1f} meses"]
        total_inv = cierre["valor_inv"].sum()
        tarjetas = []
        for k, n in enumerate(NIVELES):
            g = cierre[cierre["nivel_obs"] == k]
            cob = meses(g["Dias_Cobertura_Inventario"].median()) if len(g) else "—"
            tarjetas.append(f"""
            <div style="min-height:196px;height:100%;padding:16px 18px;background:#FFFFFF;border:1px solid #E3E7ED;border-top:4px solid {COL_NIVEL[k]};border-radius:12px;display:flex;flex-direction:column;justify-content:space-between;box-sizing:border-box;font-family:'Segoe UI',Inter,sans-serif;color:{TINTA};">
              <span style="display:flex;justify-content:space-between;"><b style="font-size:14px;">Nivel {n}</b><span style="font-size:11px;font-weight:600;color:{PILL[k][1]};">{rangos[k]}</span></span>
              <span><span style="font-size:30px;font-weight:700;color:{NAVY};">{len(g):,}</span> <span style="font-size:12px;color:{GRIS};">producto–región</span></span>
              <span style="font-size:12px;color:{GRIS};"><b style="color:{TINTA};">{dinero(g['valor_inv'].sum())}</b> valor del inventario · <b style="color:{TINTA};">{pct(g['valor_inv'].sum(), total_inv)}</b> del total</span>
              <span style="display:flex;flex-direction:column;gap:4px;border-top:1px solid #EEF1F4;padding-top:10px;font-size:12px;color:{GRIS};">
                <span>Cobertura mediana: <b style="color:{TINTA};">{cob}</b></span><span>Cumplimiento de venta: <b style="color:{TINTA};">{pct(g['units_sold'].sum(), g['units_expected'].sum())}</b></span></span>
            </div>""")
        st.html(grid(tarjetas, 4))

        # 03 · Evolución mensual
        encabezado_seccion("03", "trend", "Evolución mensual",
                           "Comportamiento de las ventas y del inventario en niveles Alto y Crítico a lo largo del periodo")
        por_mes = rango.groupby("t").apply(lambda x: pd.Series({
            "ventas": x["sales_amount"].sum(),
            "alto": x.loc[x["nivel_obs"] >= 2, "valor_inv"].sum() / x["valor_inv"].sum() * 100 if x["valor_inv"].sum() else 0}),
            include_groups=False).reset_index()
        por_mes["Mes"] = por_mes["t"].map(etiqueta_mes)
        div_v, suf_v = (1e6, "M") if por_mes["ventas"].max() >= 1e6 else (1e3, "K")
        izq, der = st.columns(2)
        izq.plotly_chart(grafica_linea(por_mes["Mes"], por_mes["ventas"] / div_v, NAVY, f"Ventas por mes ($ {suf_v})", "Monto de venta mensual", ",.1f"),
                         width="stretch")
        der.plotly_chart(grafica_linea(por_mes["Mes"], por_mes["alto"], COL_NIVEL[2], "Inventario en niveles Alto y Crítico por mes (%)",
                                       "Participación en el valor total del inventario de cada mes", ".1f"), width="stretch")

        # 04 · Regiones
        encabezado_seccion("04", "pin", "Desempeño por región", "Ventas, cumplimiento e inventario de cada región con su nivel de sobrestock predominante")
        izq, der = st.columns(2)
        alto_reg = cierre[cierre["nivel_obs"] >= 2].groupby("Región")["valor_inv"].sum().reindex(sorted(cierre["Región"].unique()), fill_value=0)
        izq.html(tarjeta("Inventario en niveles Alto y Crítico por región", "Valor del inventario al cierre del periodo",
                         barras_h([(r, v, dinero(v)) for r, v in alto_reg.sort_values(ascending=False).items()]), alto=440))
        res = rango.groupby("Región").agg(v=("sales_amount", "sum"), u=("units_sold", "sum"), e=("units_expected", "sum")).sort_values("v", ascending=False)
        filas = []
        for r, x in res.iterrows():
            c = cierre[cierre["Región"] == r]
            pred = int(c["nivel_obs"].mode().iloc[0]) if len(c) else 0
            filas.append([f"<b>{esc(r)}</b>", dinero(x["v"]), pct(x["u"], x["e"]), dinero(c["valor_inv"].sum()),
                          meses(c["Dias_Cobertura_Inventario"].median()) if len(c) else "—", pill(pred)])
        der.html(tarjeta("Resumen por región", "Ordenado por ventas del periodo · nivel predominante al cierre",
                         tabla([("Región", "l"), ("Ventas", "r"), ("Cumpl.", "r"), ("Inventario", "r"), ("Cobertura", "r"), ("Nivel", "l")], filas), alto=440))

        # 05 · Productos
        encabezado_seccion("05", "box", "Desempeño por producto", "Ventas, cumplimiento y concentración de sobrestock por subcategoría")
        izq, der = st.columns(2)
        sub_v = rango.groupby("Subcategoría")["sales_amount"].sum().sort_values(ascending=False)
        izq.html(tarjeta("Ventas por subcategoría", "Monto de venta acumulado en el periodo",
                         barras_h([(s, v, dinero(v)) for s, v in sub_v.items()]), alto=560))
        filas = []
        for (cat, s), x in rango.groupby(["Categoría", "Subcategoría"]).agg(v=("sales_amount", "sum"), u=("units_sold", "sum"),
                                                                            e=("units_expected", "sum")).iterrows():
            c = cierre[cierre["Subcategoría"] == s]
            filas.append((ORDEN_CATEGORIA.index(cat), -x["v"], [f'<span style="color:{GRIS};">{esc(cat)}</span>', f"<b>{esc(s)}</b>", dinero(x["v"]),
                          pct(x["u"], x["e"]), meses(c["Dias_Cobertura_Inventario"].median()) if len(c) else "—",
                          f"<b>{int((c['nivel_obs'] >= 2).sum())} de {len(c)}</b>"]))
        filas = [f for *_, f in sorted(filas, key=lambda z: (z[0], z[1]))]
        der.html(tarjeta("Detalle por subcategoría", "Producto–región en niveles Alto y Crítico al cierre del periodo",
                         tabla([("Categoría", "l"), ("Subcategoría", "l"), ("Ventas", "r"), ("Cumpl.", "r"), ("Cobertura", "r"), ("Alto + Crít.", "r")],
                               filas, alto_max=460), alto=560))

# ──────────────────────────────────────────────────────────────────────────────
# Pie de página
# ──────────────────────────────────────────────────────────────────────────────
st.html(f"""
<div style="margin-top:18px;padding:14px 18px;background:#FFFFFF;border:1px solid #E3E7ED;border-radius:12px;font-family:'Segoe UI',Inter,sans-serif;font-size:11px;color:{GRIS};line-height:1.7;">
  <div><b style="color:{TINTA};">Niveles de sobrestock</b> (meses que tardaría en venderse el inventario): Bajo ≤ {CORTES[0]:.1f} · Medio {CORTES[0]:.1f}–{CORTES[1]:.1f} · Alto {CORTES[1]:.1f}–{CORTES[2]:.1f} · Crítico &gt; {CORTES[2]:.1f}. En el Plan de acción es el nivel pronosticado para el trimestre siguiente; en el Diagnóstico, el nivel observado.</div>
  <div><b style="color:{TINTA};">Acciones:</b> Reducir compras = Alto, o Crítico con demanda esperada baja · Transferir = Crítico con demanda media o alta · Reabastecer = Bajo con demanda alta · Mantener = el resto.</div>
  <div><b style="color:{TINTA};">Cálculos:</b> ventas, unidades y cumplimiento se acumulan en el rango de fechas; el inventario (restante × precio de lista) se toma solo del último mes del rango, sin acumular meses. El plan corresponde al trimestre siguiente al último mes del rango.</div>
  <div><b style="color:{TINTA};">Fuente:</b> data_DanuStore_Equipo6_Mensual.csv. Pronóstico: red neuronal de la Etapa 5.1, que acierta 8 de cada 10 casos en datos que no vio al entrenar. Elaborado por Nexxus Consulting.</div>
</div>""")


### Tema claro
Fija el tablero en modo claro aunque el navegador esté en modo oscuro.

In [ ]:
import os
os.makedirs(".streamlit", exist_ok=True)
open(".streamlit/config.toml", "w").write('[theme]\nbase = "light"\n')
print("✔ Tema claro configurado")

## 4. Verificar archivos

In [ ]:
import pandas as pd
print("Líneas de app.py:", sum(1 for _ in open("app.py")))              # debe ser > 400
df = pd.read_csv("data_DanuStore_Equipo6_Mensual.csv")
print("Data:", df.shape)                                                 # (21840, 20)
!ls -lh /content/app.py /content/data_DanuStore_Equipo6_Mensual.csv /content/logo_danu.png 2>/dev/null

## 5. Lanzar la app
1. Corre la primera celda y **copia la IP**: es la contraseña del túnel.
2. Corre la segunda celda y abre la URL `https://...loca.lt`; pega la IP y da clic en *Click to Submit*.
3. La primera vez tarda unos segundos: entrena la red neuronal.

In [ ]:
!wget -q -O - ipv4.icanhazip.com

34.22.45.67


In [ ]:
!streamlit run app.py --server.port 8501 --server.headless true &> /content/logs.txt & npx -y localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏^C


### Alternativa si localtunnel no carga
Detén la celda anterior y corre esta. Busca la URL `https://...trycloudflare.com` (no pide contraseña).

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
!streamlit run app.py --server.port 8501 --server.headless true &> /content/logs.txt & ./cloudflared tunnel --url http://localhost:8501

2026-11-19T21:43:25Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-19T21:43:25Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-19T21:43:28Z INF +--------------------------------------------------------------------------------------------+
2026-11-19T21:43:28Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-19T21:43:28Z INF |  https://seafood-copper-talking-stopped.trycloudflare.

### Si algo falla: revisar el log

In [ ]:
!tail -40 /content/logs.txt